In [ ]:

import torch
import math
import torch.nn as nn



class LinearLayer:
    
    def __init__(self,in_features,out_features):
        self.weights = torch.randn(out_features,in_features)
        self.bias = torch.zeros(out_features)

        
    def forward(self,x):
        self.x = x
        return x @ self.weights.T + self.bias

    def backward(self,grad_out):
        grad_inputs = grad_out @ self.weights
        x_flat = self.x.reshape(-1, self.x.shape[-1])
        grad_out_flat = grad_out.reshape(-1, grad_out.shape[-1])
        self.weights.grad = grad_out_flat.T @ x_flat
        self.bias.grad = grad_out_flat.sum(dim=0)
        return grad_inputs

class MyRelu:
    def forward(self,x):
        self.x = x
        return torch.clamp(x,min=0)

    def backward(self,grad_out):
        return grad_out * (self.x > 0)


class MyMlp:
    def __init__(self,in_features,hidden_features,out_features):
        self.linear1 = LinearLayer(in_features,hidden_features)
        self.relu = MyRelu()
        self.linear2 = LinearLayer(hidden_features,out_features)

    def forward(self,x):
        self.x = x
        x =  self.linear1.forward(x)
        x =  self.relu.forward(x)
        x =  self.linear2.forward(x)
        return x

    def backward(self,grad_out):
        grad_layer2 = self.linear2.backward(grad_out)
        grad_relu = self.relu.backward(grad_layer2)
        grad_layer1 = self.linear1.backward(grad_relu)
        return grad_layer1

    def parameters(self):
        return [
            self.linear1.weights,self.linear1.bias,self.linear2.weights,self.linear2.bias
        ]

class MyMSELoss:
    def forward(self,y_pred,y_true):
        self.y_pred = y_pred
        self.y_true = y_true
        return torch.mean((y_pred-y_true)**2)

    def backward(self):
        N = self.y_pred.numel()
        return (2/N)*(self.y_pred - self.y_true)


class MySgd:
    def __init__(self,params,lr):
        self.params = list(params)
        self.lr = lr

    
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

    def step(self):
        with torch.no_grad():
            for p in self.params:
                if p.grad is not None:
                    p-=self.lr*p.grad

in_features=1000
out_features=10
hidden_features=128
lr = 0.001

X = torch.randn(32,1000)
Y = torch.randn(32,10)

model = MyMlp(in_features,hidden_features,out_features)
loss_fn = MyMSELoss()

params = list(model.parameters())
optim = MySgd(params,lr)

for epoch in range(100):

    optim.zero_grad()
    
    # forward pass
    y_pred = model.forward(X)

    # loss
    loss = loss_fn.forward(y_pred,Y)

    #backprop

    grad_out = loss_fn.backward()
    model.backward(grad_out)
    
    optim.step()
    
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


### Why We Need an Optimizer

so optimizer's role is to update the weights and instead of writing the boiler plate again and again
and writing the full length model.`linear1`.self.`grad_weights` we can use the parameters as the input and then it does the work

we need to use the custom parameter method because we aren't using the `nn.Module`s params so we need to inherit it
and at the step of the optimizer, we step through each parameter and update it

so the first step is making the gradients to zero and after that we just need to make sure them not accumulating,so we use the zero grad to make them zero
and next step is to update the weights and biases

### Custom vs. `nn.Module` Parameters

as we have writing the custom modules again,because we can see the workflow clearly and we can upgrade to the torch modules
so if you have used the `nn.Parameter` and then you can see this
if you still want everything without the `nn.Parameter` then you can check the Raw_MLP.ipynb file

